# Es Freschers $ \alpha $

In [1]:
#librerie
import numpy as np
from scipy.linalg import expm
import matplotlib.pyplot as plt
from qutip import *
from IPython.display import Image, display, Math

In [2]:
#funzione per plottare in LaTex delle matrici
def array_to_latex(array, real = False, array_name = None):
    array = array.real if real else array
    matrix = ''
    for row in array:
        try:
            for number in row:
                matrix += f'{number}&'
        except TypeError:
            matrix += f'{row}&'
        matrix = matrix[:-1] + r'\\'
    if array_name != None:
        display(Math(array_name+r' = \begin{bmatrix}'+matrix+r'\end{bmatrix}'))
    else:
        display(Math(r'\begin{bmatrix}'+matrix+r'\end{bmatrix}'))

## Excitonic Dimer
We will use a excionic dimer as model system; each monomer is a Two_Level_system (TLS), with both the ground state $ \ket{g} = 0.0 Ev $ and an excited state with differente energies, in particular : $ \ket{e_1} = 1.46 Ev $ and $ \ket{e_2} = 1.55 Ev $. We can now define the system's Hamiltonian in the $ \textbf{Site Basis} $ as : $$ H = H_1 + H_2 + H_{int} $$ <center> with: $ H_1 = E_1\ket{e_1}\bra{e_1} $, $ H_2 = E_2\ket{e_2}\bra{e_2} $, $ H_{int} \propto V $ with V the Interction Potential between the two monomers.</center> <br>
To be more precise, specifing the two different Hilbet Spaces : $ H = H_1 \otimes \mathbb{1_2} +  \mathbb{1_1} \otimes H_2 + V (\ket{e_1 g_2}\bra{e_2 g_1} + \ket{e_2 g_1}\bra{e_1 g_2})$ with $ \ket{\psi_2, \psi_1} = \ket{\psi_2} \otimes \ket{\psi_1} $

In [3]:
#Hamiltonian definition
N = 2
#E = 1.5 + np.random.randn(N)*0.1 #inizializzazione casuale dell'energia dei siti
E1 = 1.46
E2 = 1.55
V = 0.1

H = [ [0, 0, 0, 0],    #scritto così per convenzioe QuTip per cui la bas deve essere (|00>, |10>, |01>, |11>)
      [0, E2, V, 0],
      [0, V, E1, 0],
      [0, 0, 0, E1+E2] ]
H_np = np.array(H)
array_to_latex(H_np)

<IPython.core.display.Math object>

It's possibile to focus only on the two excited state and follow their dynamics; in this case we introduce an effective hamiltonian like this:

In [4]:
#Excited state's Hamiltonian 
E1 = 1.46
E2 = 1.55
V = 0.1

H_exc = [ [E1, V],
          [V, E2], ]
H_exc_np = np.array(H_exc)
array_to_latex(H_exc)

<IPython.core.display.Math object>

If we now diagonalize this Hamiltonian we could get the $ \textbf{Exciton basis} $ of the system; in particular the eigenvalue in the exciton basis are : $$ E_\pm = \frac{E_1 + E_2 \pm \sqrt{4V^{2} + \Delta E^{2}}}{2} $$

In [5]:
#Excited state's Hamiltonian diagonalization

#con numpy

w_np, v_np = np.linalg.eigh(H_exc) #w = autovalori, v = autovettori

#check_energy = (E1 + E2 + np.sqrt(4.0*(V**2) + ((E1-E2)**2)))/2
#w, check_energy
#prob = v**2 #quadrato dei coefficienti restituisce probabilità di occupazione dello stato eccitato 1 o 2 
Diag_exc_np = np.diag(w_np)
array_to_latex(Diag_exc_np)

<IPython.core.display.Math object>

In [6]:
#con QuTip
H1 = (E1/2) * tensor(qeye(2) - sigmaz(), qeye(2))
H2 = (E2/2) * tensor(qeye(2), qeye(2) - sigmaz())
H_int = V * (tensor(sigmap(), sigmam()) + tensor(sigmam(), sigmap()))
H_q = (H1 + H2 + H_int)

In [7]:
w, v = H_q.eigenstates()
D_q = Qobj(np.diag(w)) #matrice diagonalizzata
D_q

Quantum object: dims=[[4], [4]], shape=(4, 4), type='oper', dtype=Dense, isherm=True
Qobj data =
[[0.         0.         0.         0.        ]
 [0.         1.39534144 0.         0.        ]
 [0.         0.         1.61465856 0.        ]
 [0.         0.         0.         3.01      ]]

In [8]:
#per la matrice dei soli stati eccitati
H_exc_q =  Qobj(H_exc_np)
w, v = H_exc_q.eigenstates()
D_exc = np.diag(w)
array_to_latex(D_exc)

<IPython.core.display.Math object>

We can build up the evolution operator $ U = \exp(-i\hat{H}dt) $

In [9]:
#with the function expm which act on matrices
dt, t_final = 1e-7, 1e-5
t = np.arange(0,t_final,dt)
U_np = expm(-1.j * H_exc_np * dt)
U_q = (-1.j * H_exc_q * dt).expm()
U_q


Quantum object: dims=[[2], [2]], shape=(2, 2), type='oper', dtype=Dense, isherm=False
Qobj data =
[[ 1.000e+00-1.46e-07j -1.505e-15-1.00e-08j]
 [-1.505e-15-1.00e-08j  1.000e+00-1.55e-07j]]

or by diagonalizing the H matrix with $ D = WHW^\dagger $, than calculate the diagonale U matrix which has only diagonal element such $ U_{jj} = e^{-i E_j dt}  $ and corripsonds to the U operator in the exciton basis; then using the eigenvectors W return to the site basis $ U = W^\dagger UW $,

In [15]:
w_np, v_np = np.linalg.eigh(H_exc)
Diag_exc_np = np.diag(w_np)  # matrice diagonale degli autovalori
U_autovector = v_np @ np.diag(np.exp(-1.j * w_np * dt)) @ np.conj(v_np).T
U = Qobj(U_autovector)
U

Quantum object: dims=[[2], [2]], shape=(2, 2), type='oper', dtype=Dense, isherm=False
Qobj data =
[[ 1.00000000e+00-1.46e-07j -1.49379262e-15-1.00e-08j]
 [-1.43462719e-15-1.00e-08j  1.00000000e+00-1.55e-07j]]

#### Trotter - Suzuki decomposition
Given an Hamiltonian operator composed of the sum of hermitian matrices which doesn't necessarily commute $ \hat{H} = \sum_{\alpha}^{A} {\hat{H_\alpha}} $ we can approximate the unitary evolutio operator such as : $$ e^{-iHt} = \lim_{n \to \infty} \left( \prod_{\alpha = 1}^{A} {e^{-iH_\alpha \frac{t}{n}}} \right) + \mathcal{O} \left( A^2 \frac{t^2}{n} \right) $$
or 
$$ e^{-iHt} = \lim_{n \to \infty} \left( \prod_{\alpha = 1}^{A} {e^{-iH_\alpha \frac{t}{n}}} \right) + \mathcal{O} \left( A^2 tdt \right)$$ 
where $ dt = \frac{t}{n} $ <br>
So in our case we can rewrite the U opperator as: $$ U \approx e^{-iV\sigma_xdt} e^{-i\frac{E_1 - E_2}{2}\sigma_zdt} $$

In [20]:
U_trotter = expm(- 1.j * V * sigmax().full() * dt)
U_trotter = U_trotter @ expm(- 1.j * (E1-E2)/2 * sigmaz().full() * dt)
U_trotter = Qobj(U_trotter)
U_trotter

Quantum object: dims=[[2], [2]], shape=(2, 2), type='oper', dtype=Dense, isherm=False
Qobj data =
[[ 1.0e+00+4.5e-09j -4.5e-17-1.0e-08j]
 [ 4.5e-17-1.0e-08j  1.0e+00-4.5e-09j]]